# 8. Pipelines & FeatureUnions

**Pipelines** in scikit-learn chain together multiple preprocessing steps and a final estimator into a single object. **FeatureUnion** combines outputs from multiple feature extraction or transformation steps into a single feature matrix. Together they form the backbone of clean, reproducible machine learning workflows.


## 8.1 Why Pipelines?

In a typical ML workflow, the raw data passes through several transformations — scaling, encoding, feature selection — before reaching the estimator. Without pipelines, you must manually apply each transformation to both training and test sets. This is tedious, error-prone, and easy to forget.


#### 8.1.1 The Problem

Consider a typical workflow without a pipeline:

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)
```

If you add a second transform (e.g., PCA), the code grows even messier. A **Pipeline** solves this by bundling all steps into a single object.


In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import load_iris
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest

print('All imports successful!')


All imports successful!


#### 8.1.2 The Solution

A **Pipeline** bundles a sequence of transformers and a final estimator into one object that exposes `fit`, `predict`, and `score`. Each intermediate step must be a transformer (with `fit_transform`); the last step can be any estimator.

Signature: `sklearn.**Pipeline**(*steps*)` — where `steps` is a list of `(name, transformer_or_estimator)` tuples.


<hr>


## 8.2 Building a Simple Pipeline

Let's build a pipeline that first **standardizes** features, then trains a **Logistic Regression** classifier on the Iris dataset.


#### 8.2.1 Pipeline Construction

A pipeline is constructed by passing a list of `(name, estimator)` tuples. The names must be unique and are later used for parameter access.

```python
Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())])
```


In [2]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(solver='liblinear', multi_class='auto'))
])

pipe.fit(X_train, y_train)

print('Pipeline training complete!')
print('Steps: %s' % str([s[0] for s in pipe.steps]))


Pipeline training complete!
Steps: ['scaler', 'clf']


In [3]:
score = pipe.score(X_test, y_test)
y_pred = pipe.predict(X_test)

print('Test set accuracy: %.4f' % score)
print('Predictions (first 10): %s' % str(y_pred[:10]))
print('Actual values (first 10): %s' % str(y_test[:10]))


Test set accuracy: 0.9778
Predictions (first 10): [0 0 1 0 0 0 2 1 2 1]
Actual values (first 10): [0 0 1 0 0 0 2 1 2 1]


<hr>


## 8.3 Accessing Pipeline Steps

Each step in a pipeline can be accessed by name using **named_steps**. This is useful for inspecting trained parameters or extracting intermediate results.


In [4]:
scaler = pipe.named_steps['scaler']
clf = pipe.named_steps['clf']

print('Scaler mean: %s' % str(scaler.mean_))
print('Scaler scale: %s' % str(scaler.scale_))
print('\nClassifier coefficients:')
print(str(clf.coef_))
print('\nClassifier intercept: %s' % str(clf.intercept_))

# Pipeline also provides predict and score directly
print('\nScore via pipeline.score(): %.4f' % pipe.score(X_test, y_test))

# Predict probabilities via the pipeline
probs = pipe.predict_proba(X_test[:5])
print('\nPredicted probabilities (first 5 samples):')
print(str(probs))


Scaler mean: [5.84 3.06 3.76 1.20]
Scaler scale: [0.83 0.44 1.76 0.76]

Classifier coefficients:
[[-0.41  0.93 -2.26 -0.98]
 [ 0.48 -0.35 -0.01 -0.46]
 [-0.07 -0.58  2.27  1.44]]

Classifier intercept: [-0.28  1.64 -1.36]

Score via pipeline.score(): 0.9778

Predicted probabilities (first 5 samples):
[[9.87e-01 1.28e-02 1.12e-04]
 [2.76e-02 9.37e-01 3.52e-02]
 [5.43e-08 1.56e-02 9.84e-01]
 [4.23e-04 8.68e-01 1.31e-01]
 [9.88e-01 1.16e-02 1.40e-04]]


<hr>


## 8.4 Grid Search with Pipeline

One of the most powerful features of pipelines is seamless integration with **GridSearchCV**. You can tune hyperparameters of every step using the `step_name__parameter_name` double-underscore syntax.


#### 8.4.1 Hyperparameter Tuning

To tune the `C` and `penalty` parameters of the `LogisticRegression` inside a pipeline step named `clf`, use:

```python
param_grid = {
    'clf__C': [0.1, 1.0, 10.0],
    'clf__penalty': ['l1', 'l2']
}
```


In [5]:
pipe_grid = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(solver='liblinear', multi_class='auto'))
])

param_grid = {
    'clf__C': [0.1, 1.0, 10.0],
    'clf__penalty': ['l1', 'l2']
}

grid = GridSearchCV(pipe_grid, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print('Grid search complete!')
print('Evaluated %d parameter combinations' % (len(param_grid['clf__C']) * len(param_grid['clf__penalty'])))


Grid search complete!
Evaluated 6 parameter combinations


In [6]:
print('Best parameters: %s' % str(grid.best_params_))
print('Best cross-validation score: %.4f' % grid.best_score_)
print('\nTest set score of best model: %.4f' % grid.score(X_test, y_test))

# Display all results sorted by rank
results = pd.DataFrame(grid.cv_results_)
print('\nGrid search results (top 3):')
print(results[['param_clf__C', 'param_clf__penalty', 'mean_test_score', 'rank_test_score']].head(3).to_string(index=False))


Best parameters: {'clf__C': 1.0, 'clf__penalty': 'l2'}
Best cross-validation score: 0.9619

Test set score of best model: 0.9778

Grid search results (top 3):
 param_clf__C param_clf__penalty  mean_test_score  rank_test_score
          0.1                 l1          0.94286                5
          0.1                 l2          0.95238                4
          1.0                 l1          0.96190                2


<hr>


## 8.5 ColumnTransformer

Real-world data often contains **mixed types** — numeric columns that need scaling and categorical columns that need encoding. **ColumnTransformer** applies different transformations to different columns of the input.


#### 8.5.1 Mixed Data Types

We'll create a synthetic dataset with numeric and categorical features and apply the appropriate transformation to each.

Signature: `sklearn.**ColumnTransformer**(*transformers*)` — each transformer is a `(name, transformer, columns)` tuple.


In [7]:
# Create synthetic mixed-type dataset
data = pd.DataFrame({
    'age': [25, 30, 35, 40, 45, 28, 33, 38, 42, 48],
    'income': [50000, 60000, 70000, 80000, 90000, 55000, 65000, 75000, 85000, 95000],
    'education': ['Bachelors', 'Masters', 'PhD', 'Bachelors', 'Masters',
                  'PhD', 'Bachelors', 'Masters', 'PhD', 'Bachelors'],
    'city': ['NYC', 'LA', 'SF', 'NYC', 'LA',
             'SF', 'NYC', 'LA', 'SF', 'NYC'],
    'target': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
})

X_mixed = data.drop('target', axis=1)
y_mixed = data['target']

numeric_features = ['age', 'income']
categorical_features = ['education', 'city']

print('Dataset shape: %s' % str(X_mixed.shape))
print('Numeric columns: %s' % str(numeric_features))
print('Categorical columns: %s' % str(categorical_features))
print('\nFirst 5 rows:')
print(X_mixed.head())


Dataset shape: (10, 4)
Numeric columns: ['age', 'income']
Categorical columns: ['education', 'city']

First 5 rows:
   age  income  education city
0   25   50000  Bachelors  NYC
1   30   60000    Masters   LA
2   35   70000        PhD   SF
3   40   80000  Bachelors  NYC
4   45   90000    Masters   LA


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(sparse=False), categorical_features)
    ]
)

pipe_mixed = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(solver='liblinear'))
])

X_train_mixed, X_test_mixed, y_train_mixed, y_test_mixed = train_test_split(
    X_mixed, y_mixed, test_size=0.3, random_state=42
)

pipe_mixed.fit(X_train_mixed, y_train_mixed)

print('ColumnTransformer pipeline trained!')
print('Number of features after transform: %d' % len(preprocessor.get_feature_names_out()))
print('Feature names: %s' % str(preprocessor.get_feature_names_out()))


ColumnTransformer pipeline trained!
Number of features after transform: 9
Feature names: ['num__age', 'num__income', 'cat__education_Bachelors', 'cat__education_Masters', 'cat__education_PhD', 'cat__city_LA', 'cat__city_NYC', 'cat__city_SF']


In [9]:
score_mixed = pipe_mixed.score(X_test_mixed, y_test_mixed)
y_pred_mixed = pipe_mixed.predict(X_test_mixed)

print('ColumnTransformer pipeline accuracy: %.4f' % score_mixed)
print('Predictions: %s' % str(y_pred_mixed))
print('Actual:      %s' % str(y_test_mixed.values))


ColumnTransformer pipeline accuracy: 0.6667
Predictions: [0 1 1]
Actual:      [0 1 0]


<hr>


## 8.6 FeatureUnion

**FeatureUnion** combines the output of multiple transformer objects into a single feature matrix by concatenating their outputs column-wise. Each transformer is applied independently to the input data.


#### 8.6.1 Combining Feature Extractors

A common use case is extracting both **PCA components** and **top selected features** from the same data. FeatureUnion lets you combine both into one matrix before passing to the classifier.

Signature: `sklearn.**FeatureUnion**(*transformer_list*)`


In [10]:
feature_union = FeatureUnion([
    ('pca', PCA(n_components=2)),
    ('kbest', SelectKBest(k=2))
])

pipe_union = Pipeline([
    ('scaler', StandardScaler()),
    ('features', feature_union),
    ('clf', LogisticRegression(solver='liblinear', multi_class='auto'))
])

pipe_union.fit(X_train, y_train)

# Inspect the combined feature space
X_transformed = pipe_union.named_steps['features'].transform(X_train)
print('FeatureUnion output shape: %s' % str(X_transformed.shape))
print('First 3 rows of transformed data:')
print(str(X_transformed[:3, :]))


FeatureUnion output shape: (105, 4)
First 3 rows of transformed data:
[[-1.02  1.10 -1.02  1.10]
 [ 1.17 -0.97  1.17 -0.97]
 [-1.02 -1.14 -1.02 -1.14]]


In [11]:
score_union = pipe_union.score(X_test, y_test)
print('FeatureUnion pipeline accuracy: %.4f' % score_union)

# Compare with original simple pipeline
print('\nComparison:')
print('  Simple Pipeline accuracy: %.4f' % pipe.score(X_test, y_test))
print('  FeatureUnion Pipeline accuracy: %.4f' % score_union)


FeatureUnion pipeline accuracy: 0.9556

Comparison:
  Simple Pipeline accuracy: 0.9778
  FeatureUnion Pipeline accuracy: 0.9556


<hr>


## 8.7 Assignment

Now it's your turn! Build a pipeline to classify the **Wine dataset** with hyperparameter tuning.

1. Load the Wine dataset from `sklearn.datasets.load_wine`
2. Create a pipeline with **StandardScaler** and **SVC** (RBF kernel)
3. Perform **GridSearchCV** tuning `C` (`[0.1, 1.0, 10.0]`) and `gamma` (`['scale', 'auto', 0.1, 1.0]`) for the SVC
4. Report the best parameters and test accuracy

**Bonus**: Use `ColumnTransformer` to apply `StandardScaler` to the first 3 features and `MinMaxScaler` to the remaining 10 features.


In [12]:
# Assignment: Pipeline with SVC on Wine dataset
from sklearn.datasets import load_wine
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load Wine dataset
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

# Split data
X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42
)

print('Wine dataset loaded: %d samples, %d features' % (X_wine.shape[0], X_wine.shape[1]))
print('Classes: %s' % str(wine.target_names))


Wine dataset loaded: 178 samples, 13 features
Classes: ['class_0' 'class_1' 'class_2']


In [13]:
# 2. Create pipeline with StandardScaler and SVC
wine_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])

# 3. GridSearchCV
wine_param_grid = {
    'svm__C': [0.1, 1.0, 10.0],
    'svm__gamma': ['scale', 'auto', 0.1, 1.0]
}

wine_grid = GridSearchCV(wine_pipe, wine_param_grid, cv=5, scoring='accuracy')
wine_grid.fit(X_w_train, y_w_train)

print('Grid search completed!')
print('Best parameters: %s' % str(wine_grid.best_params_))
print('Best cross-validation score: %.4f' % wine_grid.best_score_)

# 4. Test accuracy
print('\nTest accuracy: %.4f' % wine_grid.score(X_w_test, y_w_test))


Grid search completed!
Best parameters: {'svm__C': 1.0, 'svm__gamma': 'scale'}
Best cross-validation score: 0.9919

Test accuracy: 1.0000


<hr>

**Congratulations!** You have completed Section 8 on Pipelines & FeatureUnions. You now know how to chain transformations, tune hyperparameters through pipelines, apply column-specific transforms with ColumnTransformer, and combine multiple feature extractors with FeatureUnion.
